# Vector experimentation

Let's import the necessary libraries:

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams, RecommendQuery, RecommendInput, RecommendStrategy

load_dotenv("../../_starter/.env")

True

# Initialize the clients

In [7]:
embedding_client = OpenAI(
    base_url = os.getenv("AZURE_COGNITIVE_ENDPOINT"),
    api_key = os.getenv("AZURE_COGNITIVE_KEY"),
)

qdrant_client = QdrantClient(url="http://localhost:6333")

# Data set

In [107]:
def get_vector(text):
    response = embedding_client.embeddings.create(
        input=text,
        model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
    )
    return response.data[0].embedding

fruits = ["apple", "banana", "cherry", "date", "elderberry", "fig", "grape", "honeydew", "kiwi", "lemon", "mango", "nectarine", "orange", "pear", "quince", "raspberry", "strawberry", "tangerine", "ugli", "vanilla", "watermelon"]
household_items = ["chair", "table", "desk", "lamp", "couch", "bed", "bookshelf", "mirror", "picture", "clock", "vase", "plant", "rug", "curtain", "window", "door", "key", "computer", "phone", "lock", "drawer", "box"]
professions = ["king", 'queen', 'prince', 'princess', 'doctor', 'nurse', 'teacher', 'student', 'engineer', 'scientist', 'artist', 'writer', 'musician', 'actor', 'actress', 'programmer', 'designer', 'manager', 'salesperson', 'receptionist', 'secretary', 'administrator', 'accountant', 'lawyer', 'judge', 'police', 'firefighter', 'doctor', 'nurse', 'teacher', 'student', 'engineer', 'scientist', 'artist', 'writer', 'musician', 'actor', 'actress', 'programmer', 'designer', 'manager', 'salesperson', 'receptionist', 'secretary', 'administrator', 'accountant', 'lawyer', 'judge', 'police', 'firefighter']
all_items = fruits + household_items + professions

In [108]:
all_vectors = [get_vector(item) for item in all_items]

all_vectors

[[-0.020793478935956955,
  0.013975388370454311,
  -0.0008183403988368809,
  0.018724307417869568,
  -0.008179164491593838,
  -0.002002389868721366,
  0.003758853767067194,
  0.021658459678292274,
  0.016044560819864273,
  0.031834714114665985,
  0.01765580102801323,
  -0.02242167852818966,
  0.014060190878808498,
  0.0171809084713459,
  -0.017291150987148285,
  0.022319916635751724,
  0.02233687788248062,
  -0.012372628785669804,
  -0.02075955830514431,
  -0.009752243757247925,
  0.016027599573135376,
  -0.053323570638895035,
  0.013559858314692974,
  0.028595274314284325,
  0.01679929904639721,
  0.004837960936129093,
  -0.00361257023178041,
  -0.00040996522875502706,
  -0.0037334130611270666,
  0.0257798433303833,
  0.03551512584090233,
  0.031037574633955956,
  -0.009090786799788475,
  0.005609659943729639,
  -0.05518921837210655,
  0.036804117262363434,
  0.007267541252076626,
  -0.00201723026111722,
  0.023490186780691147,
  -0.027119716629385948,
  0.010981873609125614,
  -0.010

# Create and add points to collection

In [109]:
qdrant_client.create_collection(
    collection_name="items",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

qdrant_client.upsert(
    collection_name="items",
    points=[
        PointStruct(
            id=i,
            vector=all_vectors[i],
            payload={"name": all_items[i]}
        )
        for i in range(len(all_items))
    ]
)    

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

# Default behaviour

In [110]:
search_string = "apple"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

query_vector = response.data[0].embedding

In [111]:
qdrant_client.search(
    collection_name="items",
    query_vector=query_vector,
    limit=20,
)

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_54195/3048629267.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_client.search(


[ScoredPoint(id=0, version=0, score=0.9999991, payload={'name': 'apple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=10, version=0, score=0.46678716, payload={'name': 'mango'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1, version=0, score=0.46191418, payload={'name': 'banana'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=12, version=0, score=0.45882142, payload={'name': 'orange'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=38, version=0, score=0.440947, payload={'name': 'computer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=13, version=0, score=0.43914914, payload={'name': 'pear'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=0, score=0.42552352, payload={'name': 'cherry'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=9, version=0, score=0.41837722, payload={'name': 'lemon'}, vector=None, shard_key=None, order_value=None),
 ScoredPoi

# Vector substraction (something without something)

In [112]:
search_string = "apple"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

apple_vector = response.data[0].embedding

In [113]:
search_string = "fruit"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

fruit_vector = response.data[0].embedding

In [114]:
# Subtract the fruit vector from the apple vector
fruit_minus_apple_vector = [y-x for x, y in zip(fruit_vector, apple_vector)]

# Perform a similarity search on the collection
qdrant_client.search(
    collection_name="items",
    query_vector=fruit_minus_apple_vector,
    limit=5,
)

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_54195/3331891454.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_client.search(


[ScoredPoint(id=0, version=0, score=0.46311516, payload={'name': 'apple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=38, version=0, score=0.13461035, payload={'name': 'computer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=39, version=0, score=0.13391656, payload={'name': 'phone'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=81, version=0, score=0.0999966, payload={'name': 'programmer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=58, version=0, score=0.0999966, payload={'name': 'programmer'}, vector=None, shard_key=None, order_value=None)]

# Subtract

In [118]:
search_string = "apple"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

apple_vector = response.data[0].embedding

In [119]:
search_string = "mango"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

fruit_vector = response.data[0].embedding

In [120]:
# Subtract the fruit vector from the apple vector
fruit_minus_apple_vector = [y-x for x, y in zip(fruit_vector, apple_vector)]

# Perform a similarity search on the collection
qdrant_client.search(
    collection_name="items",
    query_vector=fruit_minus_apple_vector,
    limit=60,
)

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_54195/57774959.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_client.search(


[ScoredPoint(id=0, version=0, score=0.5163275, payload={'name': 'apple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=38, version=0, score=0.18212236, payload={'name': 'computer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=39, version=0, score=0.14145285, payload={'name': 'phone'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=22, version=0, score=0.13538821, payload={'name': 'table'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=24, version=0, score=0.12164587, payload={'name': 'lamp'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=35, version=0, score=0.11945393, payload={'name': 'window'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=53, version=0, score=0.10865822, payload={'name': 'artist'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=76, version=0, score=0.10865822, payload={'name': 'artist'}, vector=None, shard_key=None, order_value=None),
 Scor

# Addition

In [121]:
def get_string_vector(search_string):
    response = embedding_client.embeddings.create(
        input=search_string,
        model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
    )
    return response.data[0].embedding

In [122]:
vector_1 = get_string_vector("apple")
vector_2 = get_string_vector("orange")

In [123]:
# Subtract the fruit vector from the apple vector
vector_1_minus_2 = [x+y for x, y in zip(vector_1, vector_2)]

# Perform a similarity search on the collection
qdrant_client.search(
    collection_name="items",
    query_vector=vector_1_minus_2,
    limit=60,
)

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_54195/2138882893.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_client.search(


[ScoredPoint(id=12, version=0, score=0.8540555, payload={'name': 'orange'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=0, version=0, score=0.8540429, payload={'name': 'apple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=17, version=0, score=0.5540875, payload={'name': 'tangerine'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=10, version=0, score=0.55283016, payload={'name': 'mango'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1, version=0, score=0.538519, payload={'name': 'banana'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=9, version=0, score=0.52523005, payload={'name': 'lemon'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=0, score=0.5136428, payload={'name': 'cherry'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=16, version=0, score=0.48584482, payload={'name': 'strawberry'}, vector=None, shard_key=None, order_value=None),
 Score

# Average

In [124]:
def get_string_vector(search_string):
    response = embedding_client.embeddings.create(
        input=search_string,
        model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
    )
    return response.data[0].embedding

In [125]:
vector_1 = get_string_vector("apple")
vector_2 = get_string_vector("computer")

In [126]:
# Subtract the fruit vector from the apple vector
vector_1_minus_2 = [(x+y)/2 for x, y in zip(vector_1, vector_2)]

# Perform a similarity search on the collection
qdrant_client.search(
    collection_name="items",
    query_vector=vector_1_minus_2,
    limit=60,
)

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_54195/1435304551.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_client.search(


[ScoredPoint(id=0, version=0, score=0.84886086, payload={'name': 'apple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=38, version=0, score=0.8488072, payload={'name': 'computer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=39, version=0, score=0.56380475, payload={'name': 'phone'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=23, version=0, score=0.5064532, payload={'name': 'desk'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=42, version=0, score=0.48413652, payload={'name': 'box'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=35, version=0, score=0.47831857, payload={'name': 'window'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=87, version=0, score=0.4767284, payload={'name': 'administrator'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=64, version=0, score=0.4767284, payload={'name': 'administrator'}, vector=None, shard_key=None, order_value=Non

# Combining operations

In [91]:
def get_string_vector(search_string):
    response = embedding_client.embeddings.create(
        input=search_string,
        model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
    )
    return response.data[0].embedding

In [142]:
vector_1 = get_string_vector("king")
vector_2 = get_string_vector("man")
vector_3 = get_string_vector("woman")

In [143]:
# Subtract the fruit vector from the apple vector
queen_maybe = [x-y+z for x, y, z in zip(vector_1, vector_2, vector_3)]

# Perform a similarity search on the collection
results = qdrant_client.search(
    collection_name="items",
    query_vector=queen_maybe,
    limit=5,
)

print(results[0].score - results[1].score)

results

0.16901792000000004


/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_54195/779513279.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=43, version=0, score=0.69037426, payload={'name': 'king'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=44, version=0, score=0.52135634, payload={'name': 'queen'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=46, version=0, score=0.42100483, payload={'name': 'princess'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=80, version=0, score=0.3656287, payload={'name': 'actress'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=57, version=0, score=0.3656287, payload={'name': 'actress'}, vector=None, shard_key=None, order_value=None)]

# Combining operations 2

In [144]:
def get_string_vector(search_string):
    response = embedding_client.embeddings.create(
        input=search_string,
        model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
    )
    return response.data[0].embedding

In [162]:
vector_1 = get_string_vector("king")
vector_2 = get_string_vector("male sovereign agnatic His Majesty cyning")
vector_3 = get_string_vector("woman")

In [165]:
# Subtract the fruit vector from the apple vector
queen_maybe = [x-y+z for x, y, z in zip(vector_1, vector_2, vector_3)]

# Perform a similarity search on the collection
results = qdrant_client.search(
    collection_name="items",
    query_vector=queen_maybe,
    limit=5,
)

print(results[0].score - results[1].score)

results

0.07812733000000005


/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_54195/779513279.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=43, version=0, score=0.55308557, payload={'name': 'king'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=44, version=0, score=0.47495824, payload={'name': 'queen'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=76, version=0, score=0.4074124, payload={'name': 'artist'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=53, version=0, score=0.4074124, payload={'name': 'artist'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=57, version=0, score=0.40037376, payload={'name': 'actress'}, vector=None, shard_key=None, order_value=None)]

Now let's perform a similarity search on the collection from the previous example.

In [5]:
collection_name = "my_random_items"

# Perform a similarity search on the collection
results = qdrant_client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=10,
)

results

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_51017/3219476414.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=0, version=0, score=0.9999991, payload={'text': 'apple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4, version=4, score=0.46746942, payload={'text': 'pineapple'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1, version=1, score=0.46200228, payload={'text': 'banana'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=2, score=0.45880446, payload={'text': 'orange'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=12, version=12, score=0.44105482, payload={'text': 'computer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=3, score=0.4390483, payload={'text': 'pear'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=9, version=9, score=0.42280066, payload={'text': 'tree'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=13, version=13, score=0.41731912, payload={'text': 'phone'}, vector=None, shard_key=None, order_value=None),
 Scored

As expected, the apple is the most similar item to our query.

Let's try to do it for a different query.

Results should be different now but similar to the new query.

In [6]:
search_string = "computer"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

query_vector = response.data[0].embedding
print(query_vector[0:5])

[-0.005014487542212009, 0.013716224581003189, -0.006623391527682543, -0.012137194164097309, -0.00048517834511585534]


In [7]:
collection_name = "my_random_items"

# Perform a similarity search on the collection
results = qdrant_client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=10,
)

results

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_51017/3219476414.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=12, version=12, score=0.999999, payload={'text': 'computer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14, version=14, score=0.6173243, payload={'text': 'laptop'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=16, version=16, score=0.60072935, payload={'text': 'keyboard'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=18, version=18, score=0.56812644, payload={'text': 'printer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=13, version=13, score=0.53987837, payload={'text': 'phone'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=15, version=15, score=0.5165946, payload={'text': 'mouse'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=7, version=7, score=0.4878015, payload={'text': 'car'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=17, version=17, score=0.48593488, payload={'text': 'monitor'}, vector=None, shard_key=None, order_value=None)

# Find dissimilar items
Let's flips this on its head and find the most dissimilar items now.

First lets generate the new vector first.

In [7]:
search_string = "banana"

response = embedding_client.embeddings.create(
    input=search_string,
    model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
)

query_vector = response.data[0].embedding
print(query_vector[0:5])

[-0.007021163124591112, -0.00937414076179266, -0.006371545139700174, 0.026437947526574135, -0.030305441468954086]


We can get the all 20 points from the collection and keep only the last 5.

In [8]:
results = qdrant_client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=20,
)

reversed_results = results[-5:]
reversed_results

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_9383/4115420862.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=19, version=19, score=0.25592452, payload={'text': 'chair'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=8, version=8, score=0.2523036, payload={'text': 'house'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14, version=14, score=0.2510685, payload={'text': 'laptop'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=17, version=17, score=0.23831232, payload={'text': 'monitor'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=18, version=18, score=0.23101115, payload={'text': 'printer'}, vector=None, shard_key=None, order_value=None)]

This give us pretty good results, as a chair is not very similar to a banana.

However this oparation might be costly and time consuming on larger datasets.

# Experimentation

This is where I encourage you to experiment with the data, vectors and queries on your own.

For example we can try to reverse each element of the vector and try to search again.

This is very naive approach and might not work well but let's try it anyway.

In [9]:
reverse_query_vector = [-x for x in query_vector]

print(reverse_query_vector[0:5])

[0.007021163124591112, 0.00937414076179266, 0.006371545139700174, -0.026437947526574135, 0.030305441468954086]


Let's try to find the most simillar items to this new vector.

In [10]:
results = qdrant_client.search(
    collection_name=collection_name,
    query_vector=reverse_query_vector,
    limit=5,
)

results

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_9383/340927645.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=18, version=18, score=-0.23101115, payload={'text': 'printer'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=17, version=17, score=-0.23831232, payload={'text': 'monitor'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14, version=14, score=-0.2510685, payload={'text': 'laptop'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=8, version=8, score=-0.2523036, payload={'text': 'house'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=19, version=19, score=-0.25592452, payload={'text': 'chair'}, vector=None, shard_key=None, order_value=None)]

Results are decent, but not perfect, try to find a better way to do this.